# NETWORK MODELING FOR INFECTIOUS DISEASE SPREAD

- Data Collection, Cleaning and Agglomeration
- Sources
  - 1) CDC-NNDSSAPI(National Notifiable Diseases Surveillance System)- https://data.cdc.gov/NNDSS/NNDSS-Weekly-Data/x9gk-5huc/data_preview
  - 2) Census.gov API- https://www.census.gov/data/tables/time-series/demo/popest/2020s-counties-total.html
  - 3) BEAAPI- https://bea.gov/data/gdp/gdp-by-county

# CDC NNDSS weekly county-level data

In [ ]:
import os
import pandas as pd

# CDC NNDSS weekly county-level data

# Path
cdc_file = os.path.join(os.getcwd(), "NNDSS_Weekly_Data_20260524.csv")
# Data Importation
df_cdc = pd.read_csv(cdc_file)
# Data Overview
print(df_cdc.shape)
df_cdc.head()

(1840440, 16)


,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,"Current week, flag",Previous 52 week Max,"Previous 52 weeks Max, flag",Cumulative YTD Current MMWR Year,"Cumulative YTD Current MMWR Year, flag",Cumulative YTD Previous MMWR Year,"Cumulative YTD Previous MMWR Year, flag",LOCATION1,LOCATION2,sort_order,geocode
0,US RESIDENTS,2022,1,Anthrax,NaN,-,0,-,NaN,-,NaN,-,NaN,US RESIDENTS,2.022010e+10,NaN
1,NEW ENGLAND,2022,1,Anthrax,NaN,-,0,-,NaN,-,NaN,-,NaN,NEW ENGLAND,2.022010e+10,NaN
2,CONNECTICUT,2022,1,Anthrax,NaN,-,0,-,NaN,-,NaN,-,CONNECTICUT,NaN,2.022010e+10,POINT (-72.738288 41.575155)
3,MAINE,2022,1,Anthrax,NaN,-,0,-,NaN,-,NaN,-,MAINE,NaN,2.022010e+10,POINT (-69.06137 45.117911)
4,MASSACHUSETTS,2022,1,Anthrax,NaN,-,0,-,NaN,-,NaN,-,MASSACHUSETTS,NaN,2.022010e+10,POINT (-71.481104 42.151077)


In [ ]:
# Checking number and percentage of missing values in each column
print(df_cdc.isnull().sum())


Reporting Area                                   0
Current MMWR Year                                0
MMWR WEEK                                        0
Label                                            0
Current week                               1650080
Current week, flag                           67439
Previous 52 week Max                        206629
Previous 52 weeks Max, flag                 535425
Cumulative YTD Current MMWR Year           1218675
Cumulative YTD Current MMWR Year, flag      202897
Cumulative YTD Previous MMWR Year          1173087
Cumulative YTD Previous MMWR Year, flag     234134
LOCATION1                                   341796
LOCATION2                                  1498644
sort_order                                       0
geocode                                     418209
dtype: int64


In [ ]:
print((df_cdc.isnull().sum() / len(df_cdc)) * 100)

Reporting Area                              0.000000
Current MMWR Year                           0.000000
MMWR WEEK                                   0.000000
Label                                       0.000000
Current week                               89.656821
Current week, flag                          3.664287
Previous 52 week Max                       11.227152
Previous 52 weeks Max, flag                29.092228
Cumulative YTD Current MMWR Year           66.216503
Cumulative YTD Current MMWR Year, flag     11.024375
Cumulative YTD Previous MMWR Year          63.739486
Cumulative YTD Previous MMWR Year, flag    12.721632
LOCATION1                                  18.571429
LOCATION2                                  81.428571
sort_order                                  0.000000
geocode                                    22.723316
dtype: float64


In [ ]:
# Retaining only the columns with less than 25% missing values
columns_to_retain = df_cdc.columns[df_cdc.isnull().mean() < 0.25]
df_cdc = df_cdc[columns_to_retain]
# Rechecking percentage of missing values in each column
print((df_cdc.isnull().sum() / len(df_cdc)) * 100)

Reporting Area                              0.000000
Current MMWR Year                           0.000000
MMWR WEEK                                   0.000000
Label                                       0.000000
Current week, flag                          3.664287
Previous 52 week Max                       11.227152
Cumulative YTD Current MMWR Year, flag     11.024375
Cumulative YTD Previous MMWR Year, flag    12.721632
LOCATION1                                  18.571429
sort_order                                  0.000000
geocode                                    22.723316
dtype: float64


In [ ]:
# Removing rows with missing values in the dataset
df_cdc = df_cdc.dropna()
# Checking the structure and size of data
df_cdc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1064920 entries, 2 to 1840431
Data columns (total 11 columns):
 #   Column                                   Non-Null Count    Dtype  
---  ------                                   --------------    -----  
 0   Reporting Area                           1064920 non-null  object 
 1   Current MMWR Year                        1064920 non-null  int64  
 2   MMWR WEEK                                1064920 non-null  int64  
 3   Label                                    1064920 non-null  object 
 4   Current week, flag                       1064920 non-null  object 
 5   Previous 52 week Max                     1064920 non-null  object 
 6   Cumulative YTD Current MMWR Year, flag   1064920 non-null  object 
 7   Cumulative YTD Previous MMWR Year, flag  1064920 non-null  object 
 8   LOCATION1                                1064920 non-null  object 
 9   sort_order                               1064920 non-null  float64
 10  geocode                

Feature Engineering: Getting County Name from geocode variable

In [ ]:
%pip install geopy shapely


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 1.4 MB/s eta 0:00:01
   ------------------ --------------------- 0.8/1.7 MB 1.8 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 1.7 MB/s eta 0:00:01
   ------------------------------------ --- 1.6/1.7 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 1.5 MB/s  0:00:01

   ---------------------------------------- 0/3 [shapely]
   ---------------------------------------- 0/3 [shapely]
   ---------------------------------------- 0/3 [shapely]
   ---------------------------------------- 0/3 [shapely]
   ---------------------------------------- 0/3 [shapely]
   ---------------------------------------- 0/3 [shapely]
   ---------------------------------------- 0/3 [shapely]
   -----

In [ ]:

import pandas as pd
from geopy.geocoders import Nominatim
from shapely import wkt
from tqdm import tqdm
import time

# Convert POINT string into longitude/latitude
def extract_lat_lon(point_str):
    """
    Extract lon/lat from:
    POINT (-72.738288 41.575155)
    """
    try:
        geom = wkt.loads(point_str)
        return geom.y, geom.x   # latitude, longitude
    except:
        return None, None

# Create latitude/longitude features
df_cdc[["latitude", "longitude"]] = df_cdc["geocode"].apply(
    lambda x: pd.Series(extract_lat_lon(x))
)

In [ ]:
# Reverse geocoding to obtain county names
geolocator = Nominatim(user_agent="cdc_county_mapper")

def get_county(lat, lon):
    try:
        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True,
            language="en"
        )

        if location and "county" in location.raw["address"]:
            return location.raw["address"]["county"]

        return None

    except:
        return None

# Avoid repeated API calls for duplicate coordinates
unique_coords = (
    df_cdc[["latitude", "longitude"]]
    .drop_duplicates()
    .dropna()
)

coord_to_county = {}

for _, row in tqdm(unique_coords.iterrows(), total=len(unique_coords)):
    lat = row["latitude"]
    lon = row["longitude"]

    county = get_county(lat, lon)

    coord_to_county[(lat, lon)] = county

    # Respect API rate limits
    time.sleep(1)

# Map county names back to dataframe
df_cdc["county"] = df_cdc.apply(
    lambda row: coord_to_county.get(
        (row["latitude"], row["longitude"])
    ),
    axis=1
)


100%|██████████| 233/233 [05:52<00:00,  1.51s/it]


In [ ]:
#  Remove the word "County" from county names and strip any leading/trailing whitespace

df_cdc["county"] = (
    df_cdc["county"]
    .str.replace(" County", "", regex=False)
    .str.strip()
)

In [ ]:
df_cdc.head()

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,"Current week, flag",Previous 52 week Max,"Cumulative YTD Current MMWR Year, flag","Cumulative YTD Previous MMWR Year, flag",LOCATION1,sort_order,geocode,latitude,longitude,county
2,CONNECTICUT,2022,1,Anthrax,-,0,-,-,CONNECTICUT,2.022010e+10,POINT (-72.738288 41.575155),41.575155,-72.738288,Lower Connecticut River Valley Planning Region
3,MAINE,2022,1,Anthrax,-,0,-,-,MAINE,2.022010e+10,POINT (-69.06137 45.117911),45.117911,-69.061370,Piscataquis
4,MASSACHUSETTS,2022,1,Anthrax,-,0,-,-,MASSACHUSETTS,2.022010e+10,POINT (-71.481104 42.151077),42.151077,-71.481104,Worcester
5,NEW HAMPSHIRE,2022,1,Anthrax,-,0,-,-,NEW HAMPSHIRE,2.022010e+10,POINT (-71.57139 43.680429),43.680429,-71.571390,Belknap
6,RHODE ISLAND,2022,1,Anthrax,-,0,-,-,RHODE ISLAND,2.022010e+10,POINT (-71.534637 41.572574),41.572574,-71.534637,South


In [ ]:
# Printing the column names and fixing the column names for analysis
print(df_cdc.columns)
df_cdc.columns = df_cdc.columns.str.strip()
print(df_cdc.columns)

Index(['Reporting Area', 'Current MMWR Year', 'MMWR WEEK', 'Label',
       'Current week, flag', 'Previous 52 week Max',
       'Cumulative YTD Current MMWR Year, flag',
       'Cumulative YTD Previous MMWR Year, flag', 'LOCATION1', 'sort_order',
       'geocode', 'latitude', 'longitude', 'county'],
      dtype='object')
Index(['Reporting Area', 'Current MMWR Year', 'MMWR WEEK', 'Label',
       'Current week, flag', 'Previous 52 week Max',
       'Cumulative YTD Current MMWR Year, flag',
       'Cumulative YTD Previous MMWR Year, flag', 'LOCATION1', 'sort_order',
       'geocode', 'latitude', 'longitude', 'county'],
      dtype='object')


In [ ]:
# Removing commas from the "Previous 52 week Max" column
df_cdc["Previous 52 week Max"] = df_cdc["Previous 52 week Max"].str.replace(",", "")


In [ ]:
# Setting the Previous 52 Week  Max as interger variable
df_cdc["Previous 52 week Max"] = df_cdc["Previous 52 week Max"].astype(int)

In [ ]:
df_cdc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1064920 entries, 2 to 1840431
Data columns (total 14 columns):
 #   Column                                   Non-Null Count    Dtype  
---  ------                                   --------------    -----  
 0   Reporting Area                           1064920 non-null  object 
 1   Current MMWR Year                        1064920 non-null  int64  
 2   MMWR WEEK                                1064920 non-null  int64  
 3   Label                                    1064920 non-null  object 
 4   Current week, flag                       1064920 non-null  object 
 5   Previous 52 week Max                     1064920 non-null  int64  
 6   Cumulative YTD Current MMWR Year, flag   1064920 non-null  object 
 7   Cumulative YTD Previous MMWR Year, flag  1064920 non-null  object 
 8   LOCATION1                                1064920 non-null  object 
 9   sort_order                               1064920 non-null  float64
 10  geocode                

# Census county population estimates

In [ ]:
import pandas as pd

# Census county population estimates

url = "https://www2.census.gov/programs-surveys/popest/datasets/2020-2025/counties/totals/co-est2025-alldata.csv"

# Reading the full county totals file directly from Census

df_census = pd.read_csv(url, encoding="ISO-8859-1")
print(df_census.shape)
df_census.head()

(3195, 99)


,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,...,RDOMESTICMIG2021,RDOMESTICMIG2022,RDOMESTICMIG2023,RDOMESTICMIG2024,RDOMESTICMIG2025,RNETMIG2021,RNETMIG2022,RNETMIG2023,RNETMIG2024,RNETMIG2025
0,40,3,6,1,0,Alabama,Alabama,5025437,5032962,5050058,...,5.116523,5.474119,5.841064,4.854242,4.510946,5.475145,7.131285,8.317052,9.153863,6.238616
1,50,3,6,1,1,Alabama,Autauga County,58805,58914,59207,...,3.691130,8.590979,8.182476,11.756218,3.970955,3.945107,9.280274,9.878843,13.969732,5.040682
2,50,3,6,1,3,Alabama,Baldwin County,231778,233252,239450,...,29.781977,28.303847,26.276912,24.802024,22.711947,30.222000,30.739294,29.886443,29.563144,24.238166
3,50,3,6,1,5,Alabama,Barbour County,25225,24958,24510,...,-12.816366,10.313883,-1.699545,-9.593301,2.971889,-12.816366,12.181752,-0.404654,-4.552753,5.658803
4,50,3,6,1,7,Alabama,Bibb County,22285,22181,22346,...,10.824893,-14.214160,-3.146590,16.369655,3.236901,10.869809,-14.033663,-2.872974,18.138122,4.270911


In [ ]:
# Saving the Census data
df_census.to_csv("census_county_population_estimates_2020_2025.csv", index=False)

In [ ]:
# Checking missing values in the census data
print(df_census.isnull().sum())

SUMLEV         0
REGION         0
DIVISION       0
STATE          0
COUNTY         0
              ..
RNETMIG2021    0
RNETMIG2022    0
RNETMIG2023    0
RNETMIG2024    0
RNETMIG2025    0
Length: 99, dtype: int64


In [ ]:
# Checking data info
df_census.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3195 entries, 0 to 3194
Data columns (total 99 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SUMLEV                 3195 non-null   int64  
 1   REGION                 3195 non-null   int64  
 2   DIVISION               3195 non-null   int64  
 3   STATE                  3195 non-null   int64  
 4   COUNTY                 3195 non-null   int64  
 5   STNAME                 3195 non-null   object 
 6   CTYNAME                3195 non-null   object 
 7   ESTIMATESBASE2020      3195 non-null   int64  
 8   POPESTIMATE2020        3195 non-null   int64  
 9   POPESTIMATE2021        3195 non-null   int64  
 10  POPESTIMATE2022        3195 non-null   int64  
 11  POPESTIMATE2023        3195 non-null   int64  
 12  POPESTIMATE2024        3195 non-null   int64  
 13  POPESTIMATE2025        3195 non-null   int64  
 14  NPOPCHG2020            3195 non-null   int64  
 15  NPOP

In [ ]:
#  Removing the word "County" from county names and strip any leading/trailing whitespace

df_census["CTYNAME"] = (
    df_census["CTYNAME"]
    .str.replace(" County", "", regex=False)
    .str.strip()
)

In [ ]:
# Selecting POPESTIMATE columns
pop_cols = [col for col in df_census.columns if "POPESTIMATE" in col]

# Converting from wide to long

df_census_long = df_census.melt(
    id_vars=["STNAME", "CTYNAME"],
    value_vars=pop_cols,
    var_name="year",
    value_name="population_estimate"
)

# Extracting year from the 'year' column
df_census_long["year"] = df_census_long["year"].str.extract("POPESTIMATE(\d{4})").astype(int)

# Renaming columns
df_census_long = df_census_long.rename(columns={
    "STNAME": "state",
    "CTYNAME": "county"
})

# Converting state names to uppercase
df_census_long["state"] = df_census_long["state"].str.upper()

df_census_long.head()

,state,county,year,population_estimate
0,ALABAMA,Alabama,2020,5032962
1,ALABAMA,Autauga,2020,58914
2,ALABAMA,Baldwin,2020,233252
3,ALABAMA,Barbour,2020,24958
4,ALABAMA,Bibb,2020,22181


In [ ]:
df_census_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19170 entries, 0 to 19169
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   state                19170 non-null  object
 1   county               19170 non-null  object
 2   year                 19170 non-null  int64 
 3   population_estimate  19170 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 599.2+ KB


# BEA county income data

In [ ]:
import os
import requests
import pandas as pd

# 3) BEA county income data

BEA_API_KEY = "D079635D-A08D-4E4E-903B-4231899408AD"

def fetch_bea_county_income(years="2020,2021,2022,2023,2024"):
    url = "https://apps.bea.gov/api/data/"
    params = {
        "UserID": BEA_API_KEY,
        "Method": "GetData",
        "DatasetName": "Regional",
        "TableName": "CAINC1",
        "LineCode": "3",
        "GeoFips": "COUNTY",
        "Year": years,
        "ResultFormat": "JSON"
    }

    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()["BEAAPI"]["Results"]["Data"]
    return pd.DataFrame(data)

df_bea = fetch_bea_county_income()
df_bea.to_csv("bea_county_income_2020_2024.csv", index=False)
print(df_bea.shape)
df_bea.head()

(15745, 8)


,Code,GeoFips,GeoName,TimePeriod,CL_UNIT,UNIT_MULT,DataValue,NoteRef
0,CAINC1-3,01001,"Autauga, AL",2020,Dollars,0,45089,2
1,CAINC1-3,01001,"Autauga, AL",2021,Dollars,0,49254,2
2,CAINC1-3,01001,"Autauga, AL",2022,Dollars,0,49879,2
3,CAINC1-3,01001,"Autauga, AL",2023,Dollars,0,52989,2
4,CAINC1-3,01001,"Autauga, AL",2024,Dollars,0,55710,2


In [ ]:
# Creating county and state columns from GeoName

# Spliting GeoName
geo_split = df_bea["GeoName"].str.split(", ", n=1, expand=True)

# Assigning columns
df_bea["county"] = geo_split[0]
df_bea["state"] = geo_split[1]


# Mapping state abbreviations to full names
state_abbrev_to_name = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",

    # District / Territories
    "DC": "District of Columbia",
    "PR": "Puerto Rico",
    "GU": "Guam",
    "VI": "Virgin Islands",
    "AS": "American Samoa",
    "MP": "Northern Mariana Islands"
}
df_bea["state"] = df_bea["state"].map(state_abbrev_to_name)
# Converting state names to uppercase
df_bea["state"] = df_bea["state"].str.upper()
df_bea.head()

,Code,GeoFips,GeoName,TimePeriod,CL_UNIT,UNIT_MULT,DataValue,NoteRef,county,state_abbrev,state
0,CAINC1-3,01001,"Autauga, AL",2020,Dollars,0,45089,2,Autauga,AL,ALABAMA
1,CAINC1-3,01001,"Autauga, AL",2021,Dollars,0,49254,2,Autauga,AL,ALABAMA
2,CAINC1-3,01001,"Autauga, AL",2022,Dollars,0,49879,2,Autauga,AL,ALABAMA
3,CAINC1-3,01001,"Autauga, AL",2023,Dollars,0,52989,2,Autauga,AL,ALABAMA
4,CAINC1-3,01001,"Autauga, AL",2024,Dollars,0,55710,2,Autauga,AL,ALABAMA


In [ ]:
df_bea.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15745 entries, 0 to 15744
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Code          15745 non-null  object
 1   GeoFips       15745 non-null  object
 2   GeoName       15745 non-null  object
 3   TimePeriod    15745 non-null  object
 4   CL_UNIT       15745 non-null  object
 5   UNIT_MULT     15745 non-null  object
 6   DataValue     15745 non-null  object
 7   NoteRef       15745 non-null  object
 8   county        15745 non-null  object
 9   state_abbrev  15745 non-null  object
 10  state         15235 non-null  object
dtypes: object(11)
memory usage: 1.3+ MB


# Merging Datasets

Merging Datasets on year/TimePeriod, state/Reporting Area and county

In [ ]:
import pandas as pd

# Preparing CDC Data
df_cdc_merge = df_cdc.copy()

# Renaming columns for consistent merge keys
df_cdc_merge = df_cdc_merge.rename(columns={
    "Reporting Area": "state",
    "Current MMWR Year": "year",
    "Previous 52 week Max": "previous_52_week_max"
})

# Keeping only keycolumns
df_cdc_merge = df_cdc_merge[
    [
        "state",
        "county",
        "year",
        "MMWR WEEK",
        "Label",
        "previous_52_week_max",
        "latitude",
        "longitude"
    ]
]


# Cleaning merge keys


df_cdc_merge["state"] = (
    df_cdc_merge["state"]
    .astype(str)
    .str.strip()
)

df_cdc_merge["county"] = (
    df_cdc_merge["county"]
    .astype(str)
    .str.strip()
)



In [ ]:
# Preparing Census Data

df_census_merge = df_census_long.copy()

# Cleaning merge keys
df_census_merge["state"] = (
    df_census_merge["state"]
    .astype(str)
    .str.strip()
)

df_census_merge["county"] = (
    df_census_merge["county"]
    .astype(str)
    .str.strip()
)


In [ ]:
# Preparing BEA Income Data

df_bea_merge = df_bea.copy()

# Converting year
df_bea_merge["year"] = pd.to_numeric(
    df_bea_merge["TimePeriod"],
    errors="coerce"
)

# Converting income variable
df_bea_merge["income"] = pd.to_numeric(
    df_bea_merge["DataValue"].str.replace(",", ""),
    errors="coerce"
)

# Keeping only key columns
df_bea_merge = df_bea_merge[
    [
        "state",
        "county",
        "year",
        "income"
    ]
]

# Cleaning merge keys
df_bea_merge["state"] = (
    df_bea_merge["state"]
    .astype(str)
    .str.strip()
)

df_bea_merge["county"] = (
    df_bea_merge["county"]
    .astype(str)
    .str.strip()
)




In [ ]:
# Merging CDC and CENSUS datasets

df_merged = pd.merge(
    df_cdc_merge,
    df_census_merge,
    how="left",
    on=["state", "county", "year"]
)

# 5. Merging with BEA

df_merged = pd.merge(
    df_merged,
    df_bea_merge,
    how="left",   # keeps all CDC rows
    on=["state", "county", "year"]
)

# Removing Duplicates

df_merged = df_merged.loc[
    :,
    ~df_merged.columns.duplicated()
]


print(df_merged.head())



           state                                          county  year  \
0    CONNECTICUT  Lower Connecticut River Valley Planning Region  2022   
1          MAINE                                     Piscataquis  2022   
2  MASSACHUSETTS                                       Worcester  2022   
3  NEW HAMPSHIRE                                         Belknap  2022   
4   RHODE ISLAND                                           South  2022   

   MMWR WEEK    Label  previous_52_week_max   latitude  longitude  \
0          1  Anthrax                     0  41.575155 -72.738288   
1          1  Anthrax                     0  45.117911 -69.061370   
2          1  Anthrax                     0  42.151077 -71.481104   
3          1  Anthrax                     0  43.680429 -71.571390   
4          1  Anthrax                     0  41.572574 -71.534637   

   population_estimate   income  
0             176095.0      NaN  
1              17379.0  52722.0  
2             866439.0  66178.0  
3   

In [ ]:
print("\nShape:")
print(df_merged.shape)

print("\nColumns:")
print(df_merged.columns.tolist())

print("\nMissing values:")
print(df_merged.isnull().sum())

# Keeping only complete data
df_merged = df_merged.dropna()


Shape:
(1079478, 10)

Columns:
['state', 'county', 'year', 'MMWR WEEK', 'Label', 'previous_52_week_max', 'latitude', 'longitude', 'population_estimate', 'income']

Missing values:
state                        0
county                       0
year                         0
MMWR WEEK                    0
Label                        0
previous_52_week_max         0
latitude                     0
longitude                    0
population_estimate     321829
income                  353691
dtype: int64


In [ ]:
# Saving the merged dataset
df_merged.to_csv("merged_cdc_census_bea_county_data.csv", index=False)

In [ ]:
df_merged['MMWR WEEK'].value_counts()

MMWR WEEK
45    14119
46    14119
47    14119
33    14101
36    14101
35    14101
42    14101
43    14101
41    14101
38    14101
39    14101
40    14101
34    14101
37    14101
26    14098
27    14098
30    14098
32    14098
23    14097
24    14097
31    14096
25    14095
29    14076
28    14076
49    14024
48    14024
3     13964
2     13964
4     13943
5     13940
7     13940
8     13938
1     13917
22    13897
44    13896
19    13895
21    13895
20    13895
18    13895
11    13892
10    13892
13    13892
16    13891
14    13891
15    13891
6     13834
51    13803
50    13803
52    13803
9     13732
17    13685
12    12354
Name: count, dtype: int64